# Content Moderation at Scale

Companion notebook for the [Content Moderation lesson](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/21-content-moderation).

**The idea in one sentence.** Moderation is imbalanced (most content is fine), subjective
(annotators disagree), and high-stakes (both misses and false alarms hurt) — so you reach
for **focal loss** (focus training on hard examples), **inter-annotator agreement**
(Cohen's kappa), and **active learning** (label the most uncertain items first).

The three tools, from scratch:

- **Focal loss:** down-weights easy, correctly-classified examples so training focuses on
  the hard cases.
- **Cohen's kappa:** agreement between annotators *beyond chance* — the ceiling on model
  quality.
- **Active learning:** spend the labelling budget on the most *uncertain* items.

We build all three, **validate that focal loss down-weights easy negatives and kappa
behaves**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
rng = np.random.default_rng(42)

## 1 — Focal loss for extreme class imbalance

Content moderation has extreme imbalance (< 1% violation rate). Focal loss down-weights easy negatives so the model focuses on hard examples.

In [ ]:
def focal_loss(p, y, gamma=2.0, eps=1e-7):
    """
    p: predicted probabilities (N,)
    y: true labels 0/1 (N,)
    gamma: focusing parameter (0 = standard BCE, 2 = typical focal)
    """
    p = np.clip(p, eps, 1-eps)
    p_t = np.where(y == 1, p, 1-p)
    loss = -((1 - p_t)**gamma) * np.log(p_t)
    return loss.mean()

# Demonstration: easy negative vs hard negative
p_easy_neg = 0.02   # model correctly predicts ~0 for easy safe content
p_hard_neg = 0.45   # model is uncertain about borderline content

y_neg = 0  # true label: not violating
bce_easy  = -np.log(1 - p_easy_neg)
bce_hard  = -np.log(1 - p_hard_neg)
fl_easy   = ((1 - (1 - p_easy_neg))**2) * (-np.log(1 - p_easy_neg))
fl_hard   = ((1 - (1 - p_hard_neg))**2) * (-np.log(1 - p_hard_neg))

print(f"{'':20s} {'BCE loss':>12s}  {'Focal loss (γ=2)':>16s}")
print(f"Easy negative (p=0.02): {bce_easy:12.4f}  {fl_easy:16.4f}")
print(f"Hard negative (p=0.45): {bce_hard:12.4f}  {fl_hard:16.4f}")
print(f"Ratio hard/easy:        {bce_hard/bce_easy:12.1f}x  {fl_hard/fl_easy:14.1f}x")
print("Focal loss up-weights hard examples relative to easy ones")

### Validate: focal loss down-weights easy examples

Focal loss multiplies the cross-entropy by $(1-p_t)^\gamma$, so a *confidently correct*
example (high $p_t$) contributes almost nothing while a *hard* one still counts. We confirm
that raising $\gamma$ shrinks an easy example's loss far more than a hard one's — refocusing
training on what's not yet learned.

In [ ]:
def fl_single(p, y, gamma):
    return focal_loss(np.array([p]), np.array([y]), gamma=gamma)
easy = 0.02   # easy negative: model already predicts ~0 correctly (y=0)
hard = 0.45   # hard example near the boundary
for g in [0.0, 2.0]:
    print(f'gamma={g}: easy-negative loss {fl_single(easy, 0, g):.4f}, hard loss {fl_single(hard, 0, g):.4f}')
# gamma=2 shrinks the easy example far more than the hard one
ratio_easy = fl_single(easy, 0, 2.0) / fl_single(easy, 0, 0.0)
ratio_hard = fl_single(hard, 0, 2.0) / fl_single(hard, 0, 0.0)
print(f'\nfocal/BCE loss ratio -- easy: {ratio_easy:.4f}, hard: {ratio_hard:.4f}')
assert ratio_easy < ratio_hard, 'focal loss down-weights easy examples more than hard ones'
print('\n✅ focal loss refocuses training on hard examples by shrinking easy-example loss')

## 2 — Cohen's kappa for inter-annotator agreement

In [ ]:
def cohens_kappa(labels_a, labels_b):
    """
    labels_a, labels_b: arrays of 0/1 labels from two annotators
    Returns Cohen's kappa coefficient.
    """
    assert len(labels_a) == len(labels_b)
    n = len(labels_a)
    # Observed agreement
    p_o = (labels_a == labels_b).mean()
    # Expected agreement (by chance)
    p_a1 = labels_a.mean()    # annotator A's rate of labeling 1
    p_b1 = labels_b.mean()    # annotator B's rate of labeling 1
    p_e = p_a1*p_b1 + (1-p_a1)*(1-p_b1)
    return (p_o - p_e) / (1 - p_e + 1e-9)

# Simulate two annotators on 200 content items
n_items = 200
true_labels = (rng.random(n_items) < 0.1).astype(int)   # 10% violating

# Annotator A: good but noisy
a_labels = true_labels.copy()
flip_a = rng.random(n_items) < 0.1                       # 10% noise
a_labels[flip_a] = 1 - a_labels[flip_a]

# Annotator B: noisier
b_labels = true_labels.copy()
flip_b = rng.random(n_items) < 0.25                      # 25% noise
b_labels[flip_b] = 1 - b_labels[flip_b]

kappa = cohens_kappa(a_labels, b_labels)
print(f"Cohen's kappa: {kappa:.3f}")
print(f"Interpretation: {'Moderate' if 0.4 < kappa < 0.6 else 'Poor' if kappa < 0.4 else 'Substantial'} agreement")
print(f"Raw agreement: {(a_labels == b_labels).mean():.3f}")

### Validate: Cohen's kappa corrects agreement for chance

Raw agreement overstates reliability when one label dominates (two annotators who both say
"fine" 95% of the time agree 90% by luck alone). Kappa subtracts the chance agreement: it's
1 for perfect agreement, ~0 for chance-level, and negative for systematic disagreement. We
check all three regimes.

In [ ]:
a = np.array([1, 0, 1, 0, 1, 1, 0, 0])
print(f'perfect agreement kappa: {cohens_kappa(a, a.copy()):.2f}')
print(f'opposite labels  kappa: {cohens_kappa(a, 1 - a):.2f}')
# two annotators who almost always say 0 agree ~90% by chance -> kappa near 0
b1 = np.zeros(100, int); b1[:5] = 1
b2 = np.zeros(100, int); b2[95:] = 1     # both label ~5% positive, but different items
print(f'high raw-agreement, chance-level kappa: raw {np.mean(b1==b2):.2f}, kappa {cohens_kappa(b1, b2):.2f}')
assert np.isclose(cohens_kappa(a, a.copy()), 1.0), 'identical labels -> kappa 1'
assert cohens_kappa(a, 1 - a) < 0, 'opposite labels -> negative kappa'
assert cohens_kappa(b1, b2) < 0.5, 'high raw agreement from a dominant class is not high kappa'
print('\n✅ kappa corrects for chance — high raw agreement on a skewed label is not real agreement')

## 3 — Active learning: uncertainty sampling

In [ ]:
# Simulate a pool of 1000 unlabeled items with model confidence scores
n_pool = 1000
# Model outputs probabilities (simulated)
probs = rng.beta(0.5, 0.5, n_pool)  # bimodal: most items confidently classified
# Uncertainty = entropy = -p*log(p) - (1-p)*log(1-p)
eps = 1e-7
entropy = -(probs + eps)*np.log(probs + eps) - (1-probs+eps)*np.log(1-probs+eps)

# Uncertainty sampling: label the 50 most uncertain items first
k = 50
uncertain_ids = np.argsort(-entropy)[:k]
random_ids    = rng.choice(n_pool, k, replace=False)

print(f"Uncertainty-sampled items: mean entropy = {entropy[uncertain_ids].mean():.3f}")
print(f"Random-sampled items:      mean entropy = {entropy[random_ids].mean():.3f}")
print(f"Ratio: {entropy[uncertain_ids].mean()/entropy[random_ids].mean():.1f}x more informative")

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **class imbalance** | most content is fine; use focal loss / re-weighting, not plain BCE |
| **annotator disagreement** | subjective labels cap model quality; measure kappa, adjudicate |
| **precision vs recall stakes** | misses and false removals both harm; pick $F_\beta$ per policy |
| **appeals & context** | automated decisions need human review paths and context |
| **adversarial evasion** | bad actors obfuscate (leetspeak, images); moderation is a moving target |

Demo: uncertainty sampling selects more informative items than random labelling.

In [ ]:
# Active learning: with a fixed labelling budget, labelling the MOST UNCERTAIN items
# teaches the model more than random labelling, because uncertain items sit near the
# decision boundary. We compare the mean uncertainty of the selected vs a random batch.
sel_uncertainty = entropy[uncertain_ids].mean()
rand_ids = rng.choice(n_pool, size=k, replace=False)
rand_uncertainty = entropy[rand_ids].mean()
print(f'mean entropy -- uncertainty-sampled batch: {sel_uncertainty:.3f}')
print(f'mean entropy -- random batch:              {rand_uncertainty:.3f}')
assert sel_uncertainty > rand_uncertainty, 'uncertainty sampling picks higher-entropy (more informative) items'
print('\nLabelling the most uncertain items (near the boundary) is more informative per label')
print('than random sampling -> active learning stretches a limited annotation budget.')

## ✏️ Your turn

**Exercise.** Implement `f_beta(precision, recall, beta)` where $\beta > 1$ weights recall more than precision (appropriate when missing violations is more costly than a false alarm).

In [ ]:
def f_beta(precision, recall, beta=2.0):
    """F-beta score. beta > 1 weights recall more than precision."""
    # TODO(you): implement F_beta = (1 + beta^2) * P * R / (beta^2 * P + R)
    return ...

# Compare F1 (beta=1) vs F2 (beta=2) for a high-recall, low-precision model
p, r = 0.20, 0.90          # catches most violations but many false alarms
print(f"F1  (equal weight): {f_beta(p, r, beta=1.0):.4f}")
print(f"F2  (recall 4x):    {f_beta(p, r, beta=2.0):.4f}")
print(f"F0.5 (precision 4x):{f_beta(p, r, beta=0.5):.4f}")

In [ ]:
# Assertion
assert abs(f_beta(0.5, 0.5, 1.0) - 0.5) < 1e-6, "F1 of equal P=R=0.5 should be 0.5"
assert f_beta(0.2, 0.9, 2.0) > f_beta(0.2, 0.9, 1.0), "F2 should score higher-recall model higher than F1"
print(f"✓ f_beta correct: F2={f_beta(0.2, 0.9, 2.0):.4f}")

<details><summary>Solution</summary>

```python
def f_beta(precision, recall, beta=2.0):
    b2 = beta ** 2
    return (1 + b2) * precision * recall / (b2 * precision + recall + 1e-9)
```
</details>

## Key takeaways

- **Moderation is imbalanced, subjective, and high-stakes.** Accuracy misleads; you need
  the right loss, agreement metrics, and labelling strategy.
- **Focal loss focuses on hard examples** by down-weighting easy ones (verified) — vital
  when 99% of content is fine.
- **Cohen's kappa measures agreement beyond chance** (verified) — high raw agreement on a
  skewed label isn't real agreement, and annotator kappa caps model quality.
- **Active learning stretches the labelling budget** by labelling the most uncertain items
  first (demo).